In [13]:
# !pip install -q --upgrade --no-deps transformers==4.45.2 accelerate einops pillow

In [14]:
# !pip install -q --upgrade --no-deps "pillow>=10.3.0"

In [15]:
from pathlib import Path
import json
from typing import Dict, List, Tuple, Set
from collections import defaultdict

import numpy as np
import pandas as pd

PREDICTIONS_CACHE_PATH = Path("/kaggle/input/dataset-v2/predictions_cache_v1.json")
assert PREDICTIONS_CACHE_PATH.exists(), f"Không tìm thấy file: {PREDICTIONS_CACHE_PATH}"

with PREDICTIONS_CACHE_PATH.open("r", encoding="utf-8") as f:
    cache = json.load(f)

timestamp = cache.get("timestamp")
config = cache.get("config", {})
raw_predictions = cache.get("predictions", [])
errors = cache.get("errors", [])

MIN_IMPORTANCE_CORE = config.get("MIN_IMPORTANCE_CORE", 2)
original_sample_size = config.get("SAMPLE_SIZE", len(raw_predictions))

print("✅ Đọc predictions_cache.json thành công")
print(f"- Timestamp:         {timestamp}")
print(f"- Sample size (cfg): {original_sample_size}")
print(f"- Số record trong predictions: {len(raw_predictions)}")
print(f"- Số errors:         {len(errors)}")
print(f"- MIN_IMPORTANCE_CORE: {MIN_IMPORTANCE_CORE}")

✅ Đọc predictions_cache.json thành công
- Timestamp:         2025-11-14T17:15:27.241355
- Sample size (cfg): 30
- Số record trong predictions: 30
- Số errors:         0
- MIN_IMPORTANCE_CORE: 2


In [16]:
# helper vá json lỗi từ LLM
import json as _json
import re
from typing import Any


def extract_json_candidate(text: str) -> str:
    """
    Lấy đoạn JSON 'thô' từ text LLM:
    - Tìm dấu '{' đầu tiên và '}' cuối cùng.
    - Nếu không tìm được -> raise ValueError.
    """
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("Không tìm thấy JSON hợp lệ trong output của model.")
    return text[start : end + 1]


def clean_json_candidate(json_str: str) -> str:
    """
    Làm sạch một số lỗi JSON thường gặp từ LLM:
      - Bỏ BOM + strip khoảng trắng đầu/cuối.
      - Xoá comment kiểu //... và /* ... */.
      - Xoá dấu phẩy thừa trước '}' hoặc ']'.
    KHÔNG đụng chạm quá tay (vd: không auto đổi ' thành ").
    """
    # Bỏ BOM nếu có
    json_str = json_str.lstrip("\ufeff").strip()

    # Xoá comment dòng kiểu // ...
    # Ví dụ:  {  // comment }  -> { }
    json_str = re.sub(r"//.*", "", json_str)

    # Xoá comment khối kiểu /* ... */
    json_str = re.sub(r"/\*.*?\*/", "", json_str, flags=re.DOTALL)

    # Xoá dấu phẩy thừa trước } hoặc ]
    # Ví dụ: {"a": 1,} -> {"a": 1}
    #        [1, 2, ] -> [1, 2]
    json_str = re.sub(r",\s*([}\]])", r"\1", json_str)

    return json_str


def safe_json_loads_from_llm_output(output_text: str) -> Any:
    """
    Cố gắng parse JSON từ output của LLM theo 2 bước:
      1) Tách JSON candidate -> thử json.loads.
      2) Nếu lỗi -> làm sạch với clean_json_candidate -> thử lại.
    Nếu vẫn lỗi -> raise exception để code bên ngoài fallback.
    """
    # Bước 1: tách JSON candidate
    candidate = extract_json_candidate(output_text)

    # Lần 1: thử parse thẳng
    try:
        return _json.loads(candidate)
    except Exception as e_first:
        # Bước 2: làm sạch rồi thử lại
        cleaned = clean_json_candidate(candidate)
        try:
            return _json.loads(cleaned)
        except Exception as e_second:
            # Nếu muốn debug sâu, bạn có thể print 2 biến này:
            # print("===== RAW CANDIDATE =====")
            # print(candidate)
            # print("===== CLEANED CANDIDATE =====")
            # print(cleaned)
            # print("===== LỖI LẦN 1 =====", repr(e_first))
            # print("===== LỖI LẦN 2 =====", repr(e_second))
            # Ném lại lỗi lần 2 (chi tiết hơn)
            raise e_second


In [17]:
def parse_quantity_str(qty):
    """Chuyển quantity (str/float/None) -> float. Lỗi trả về 0."""
    if qty is None or qty == "":
        return 0.0
    try:
        return float(qty)
    except Exception:
        return 0.0


def extract_gt_ingredients(gt_data: Dict) -> List[Dict]:
    """Lấy list nguyên liệu GT với đầy đủ thông tin cần cho LLM & metric."""
    result = []
    for ing in gt_data.get("ingredients", []):
        result.append(
            {
                "ingredient_id": ing.get("ingredient_id"),
                "name_vi": ing.get("name_vi"),
                "name_en": ing.get("name_en"),
                "quantity": ing.get("quantity"),
                "unit": ing.get("unit"),
                "category": ing.get("category"),
                "importance": ing.get("importance", 1),
            }
        )
    return result


def extract_pred_ingredients(pred_data: Dict) -> List[Dict]:
    """Lấy list nguyên liệu từ RAG prediction."""
    result = []
    for ing in pred_data.get("ingredients", []):
        result.append(
            {
                "ingredient_id": ing.get("ingredient_id"),
                "name_vi": ing.get("name_vi"),
                "quantity": ing.get("quantity"),
                "unit": ing.get("unit"),
                "category": ing.get("category"),
            }
        )
    return result


def safe_text(x):
    if x is None:
        return ""
    return str(x)


print("✅ Định nghĩa xong helper cho dữ liệu")

✅ Định nghĩa xong helper cho dữ liệu


In [18]:
# Chuẩn hoá predictions 

samples = []
excluded_dishes = []

for item in raw_predictions:
    dish_name = item.get("dish_name", "").strip()
    gt = item.get("ground_truth") or {}
    rag = item.get("rag_prediction") or {}

    gt_ings = extract_gt_ingredients(gt)
    rag_ings = extract_pred_ingredients(rag)

    # Bỏ các món không có nguyên liệu GT hoặc không có nguyên liệu RAG
    if len(gt_ings) == 0 or len(rag_ings) == 0:
        excluded_dishes.append(
            {
                "dish_name": dish_name,
                "reason": "missing_gt_or_pred_ingredients",
            }
        )
        continue

    samples.append(
        {
            "dish_name": dish_name,
            "ground_truth": gt,
            "rag_prediction": rag,
            "gt_ingredients": gt_ings,
            "pred_ingredients": rag_ings,
        }
    )

print(f"Số sample dùng để evaluate: {len(samples)}")
print(f"Số món bị loại: {len(excluded_dishes)}")


Số sample dùng để evaluate: 30
Số món bị loại: 0


In [19]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",   # fp16/bf16 tuỳ GPU
    device_map="auto",    # để accelerate tự map lên GPU
)

print("✅ Đã load xong model:", MODEL_ID)

Device: cuda


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Đã load xong model: Qwen/Qwen2.5-3B-Instruct


In [20]:
import json as _json

def format_ingredient_line(ing: Dict, role: str) -> str:
    """
    role: "GT" hoặc "PRED"
    """
    return (
        f"{role}("
        f"id={safe_text(ing.get('ingredient_id'))}, "
        f"name_vi={safe_text(ing.get('name_vi'))}, "
        f"category={safe_text(ing.get('category'))}, "
        f"quantity={safe_text(ing.get('quantity'))} {safe_text(ing.get('unit'))}"
        f")"
    )


def build_semantic_prompt(dish_name: str, gt_ings: List[Dict], pred_ings: List[Dict]) -> str:
    """
    Tạo prompt cho LLM:
    - GT: đánh số G1..Gn
    - Pred: đánh số P1..Pm
    - YÊU CẦU: chỉ trả về các dòng dạng 'G<gt_index> P<pred_index>'
    """
    lines = [
        "Bạn là chuyên gia ẩm thực và chuyên gia đối sánh ngữ nghĩa.",
        "NHIỆM VỤ:",
        "  - So sánh NGHĨA của hai danh sách nguyên liệu (GROUND TRUTH vs RAG prediction).",
        "  - Xác định những cặp nguyên liệu tương đương về bản chất (cùng loại, khác cách gọi).",
        "",
        "QUY TẮC KHỚP NGHĨA:",
        "  • Hai nguyên liệu được xem là KHỚP nếu thực chất là cùng một loại nguyên liệu,",
        "    dù tên gọi khác nhau hoặc mô tả chi tiết hơn.",
        "  • Ví dụ: 'Gạo nếp' ~ 'Gạo nếp cái hoa vàng' → KHỚP.",
        "  • Nếu khác loại (vd: thịt bò vs thịt heo) → KHÔNG KHỚP.",
        "  • Mỗi nguyên liệu dự đoán Pj chỉ nên ghép với tối đa 1 nguyên liệu GT Gi (1–1 là lý tưởng).",
        "",
        "ĐỊNH DANH NGUYÊN LIỆU:",
        "  - G1, G2, ... = nguyên liệu Ground Truth.",
        "  - P1, P2, ... = nguyên liệu RAG prediction.",
        "",
        "CÁCH TRẢ LỜI (BẮT BUỘC):",
        "  • CHỈ TRẢ VỀ CÁC DÒNG DẠNG:",
        "        G<gt_index> P<pred_index>",
        "    Ví dụ:",
        "        G1 P3",
        "        G2 P1",
        "  • Mỗi dòng là một cặp nguyên liệu KHỚP NGHĨA.",
        "  • KHÔNG in thêm bất kỳ chữ nào, không giải thích, không JSON, không comment.",
        "  • Nếu bạn nghĩ không có cặp nào khớp, hãy trả về một dòng trống hoặc không ghi gì.",
        "",
        f"Món ăn: {dish_name}",
        "",
        "DANH SÁCH NGUYÊN LIỆU GROUND TRUTH:",
    ]

    # Gán index G1..Gn
    for idx, ing in enumerate(gt_ings, start=1):
        lines.append(
            f"  G{idx}: id={ing.get('ingredient_id')}, "
            f"name_vi={safe_text(ing.get('name_vi'))}, "
            f"category={safe_text(ing.get('category'))}, "
            f"quantity={safe_text(ing.get('quantity'))} {safe_text(ing.get('unit'))}"
        )

    lines.append("")
    lines.append("DANH SÁCH NGUYÊN LIỆU RAG PREDICTION:")

    # Gán index P1..Pm
    for idx, ing in enumerate(pred_ings, start=1):
        lines.append(
            f"  P{idx}: id={ing.get('ingredient_id')}, "
            f"name_vi={safe_text(ing.get('name_vi'))}, "
            f"category={safe_text(ing.get('category'))}, "
            f"quantity={safe_text(ing.get('quantity'))} {safe_text(ing.get('unit'))}"
        )

    lines.append("")
    lines.append("Bây giờ hãy suy nghĩ và CHỈ TRẢ VỀ các dòng dạng 'G<gt_index> P<pred_index>' như đã nêu ở trên.")

    return "\n".join(lines)

In [21]:
import re

PAIR_PATTERN = re.compile(r"\bG(\d+)\s+P(\d+)\b")


def parse_llm_pairs(output_text: str, num_gt: int, num_pred: int) -> List[Tuple[int, int]]:
    """
    Parse output từ LLM:
    - Tìm các cặp 'G<gi> P<pi>' trong text.
    - Đảm bảo:
      + 1 <= gi <= num_gt
      + 1 <= pi <= num_pred
      + Một G và một P chỉ dùng nhiều nhất 1 lần (ép 1–1).
    - Trả về list các cặp (gi_idx_0_based, pi_idx_0_based).
    """
    pairs: List[Tuple[int, int]] = []
    used_gt = set()
    used_pred = set()

    for line in output_text.splitlines():
        match = PAIR_PATTERN.search(line)
        if not match:
            continue
        gi = int(match.group(1))
        pi = int(match.group(2))

        # Kiểm tra index có hợp lệ không
        if not (1 <= gi <= num_gt and 1 <= pi <= num_pred):
            continue

        # Ép 1–1: bỏ qua nếu đã dùng
        if gi in used_gt or pi in used_pred:
            continue

        used_gt.add(gi)
        used_pred.add(pi)

        # Lưu lại index 0-based để truy cập list
        pairs.append((gi - 1, pi - 1))

    return pairs

In [22]:
def qwen_semantic_align(
    dish_name: str,
    gt_ings: List[Dict],
    pred_ings: List[Dict],
    max_new_tokens: int = 128,
) -> Dict:
    """
    Gọi Qwen (text-only) để align NGỮ NGHĨA:
    - Output model: các dòng 'G<gi> P<pi>'
    - Trả về:
      {
        "matches": [{"gt_id": str, "pred_id": str}, ...],
        "unmatched_gt_ids": [str],
        "unmatched_pred_ids": [str],
      }
    """
    if not gt_ings and not pred_ings:
        return {"matches": [], "unmatched_gt_ids": [], "unmatched_pred_ids": []}

    prompt = build_semantic_prompt(dish_name, gt_ings, pred_ings)

    messages = [
        {"role": "user", "content": prompt}
    ]

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
        )

    # Cắt phần mới sinh ra (bỏ prompt)
    new_tokens = generated_ids[0][inputs["input_ids"].shape[1]:]
    output_text = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    # Parse các cặp Gx Py
    pairs = parse_llm_pairs(output_text, num_gt=len(gt_ings), num_pred=len(pred_ings))

    # Map sang ingredient_id
    matches = []
    for gi, pi in pairs:
        gt_id = gt_ings[gi].get("ingredient_id")
        pred_id = pred_ings[pi].get("ingredient_id")
        if gt_id and pred_id:
            matches.append({"gt_id": gt_id, "pred_id": pred_id})

    # Tính unmatched dựa trên matches
    gt_ids_all = {ing.get("ingredient_id") for ing in gt_ings if ing.get("ingredient_id")}
    pred_ids_all = {ing.get("ingredient_id") for ing in pred_ings if ing.get("ingredient_id")}

    matched_gt_ids = {m["gt_id"] for m in matches}
    matched_pred_ids = {m["pred_id"] for m in matches}

    unmatched_gt_ids = list(gt_ids_all - matched_gt_ids)
    unmatched_pred_ids = list(pred_ids_all - matched_pred_ids)

    return {
        "matches": matches,
        "unmatched_gt_ids": unmatched_gt_ids,
        "unmatched_pred_ids": unmatched_pred_ids,
    }

In [23]:
def fallback_align_by_id_and_name(gt_ings: List[Dict], pred_ings: List[Dict]) -> Dict:
    """
    Fallback cực đơn giản khi LLM lỗi:
    - Ưu tiên match theo ingredient_id trùng nhau
    - Sau đó match theo name_vi (normalize lower/strip)
    """
    gt_ids = [ing.get("ingredient_id") for ing in gt_ings]
    pred_ids = [ing.get("ingredient_id") for ing in pred_ings]

    gt_used = set()
    pred_used = set()
    matches = []

    # 1. Match theo id
    id_to_gt_idx = {}
    for i, ing in enumerate(gt_ings):
        iid = ing.get("ingredient_id")
        if iid:
            id_to_gt_idx.setdefault(iid, []).append(i)

    id_to_pred_idx = {}
    for j, ing in enumerate(pred_ings):
        iid = ing.get("ingredient_id")
        if iid:
            id_to_pred_idx.setdefault(iid, []).append(j)

    common_ids = set(id_to_gt_idx.keys()) & set(id_to_pred_idx.keys())
    for iid in common_ids:
        for gi in id_to_gt_idx[iid]:
            if gi in gt_used:
                continue
            for pj in id_to_pred_idx[iid]:
                if pj in pred_used:
                    continue
                gt_used.add(gi)
                pred_used.add(pj)
                matches.append({"gt_id": gt_ings[gi]["ingredient_id"], "pred_id": pred_ings[pj]["ingredient_id"]})
                break

    # 2. Match theo name_vi (lower + strip)
    def norm_name(ing):
        return safe_text(ing.get("name_vi")).strip().lower()

    name_to_gt_idx = {}
    for i, ing in enumerate(gt_ings):
        if i in gt_used:
            continue
        name = norm_name(ing)
        if name:
            name_to_gt_idx.setdefault(name, []).append(i)

    for j, ing in enumerate(pred_ings):
        if j in pred_used:
            continue
        name = norm_name(ing)
        if not name:
            continue
        if name in name_to_gt_idx:
            # match 1-1
            gi = name_to_gt_idx[name].pop(0)
            if not name_to_gt_idx[name]:
                del name_to_gt_idx[name]
            gt_used.add(gi)
            pred_used.add(j)
            matches.append(
                {"gt_id": gt_ings[gi]["ingredient_id"], "pred_id": ing.get("ingredient_id")}
            )

    gt_all_ids = {ing.get("ingredient_id") for ing in gt_ings if ing.get("ingredient_id")}
    pred_all_ids = {ing.get("ingredient_id") for ing in pred_ings if ing.get("ingredient_id")}

    matched_gt_ids = {m["gt_id"] for m in matches}
    matched_pred_ids = {m["pred_id"] for m in matches}

    unmatched_gt_ids = list(gt_all_ids - matched_gt_ids)
    unmatched_pred_ids = list(pred_all_ids - matched_pred_ids)

    return {
        "matches": matches,
        "unmatched_gt_ids": unmatched_gt_ids,
        "unmatched_pred_ids": unmatched_pred_ids,
    }


print("✅ Định nghĩa xong fallback align (id + name)")


✅ Định nghĩa xong fallback align (id + name)


In [24]:
semantic_alignments = []

for idx, sample in enumerate(samples, start=1):
    dish_name = sample["dish_name"]
    gt_ings = sample["gt_ingredients"]
    pred_ings = sample["pred_ingredients"]

    print(f"[{idx}/{len(samples)}] Align món: {dish_name}")

    try:
        align_result = qwen_semantic_align(dish_name, gt_ings, pred_ings)
    except Exception as e:
        print(f"⚠️  Lỗi LLM cho món '{dish_name}': {e}")
        print("    → Dùng fallback align theo id + name.")
        align_result = fallback_align_by_id_and_name(gt_ings, pred_ings)

    semantic_alignments.append(align_result)

print("✅ Hoàn thành semantic align cho tất cả món")


[1/30] Align món: Phở chiên phồng bò xào
[2/30] Align món: Bún măng vịt  bằng nồi cơm điện tử
[3/30] Align món: Phở xào tim gà
[4/30] Align món: Bún xào rau cải thịt heo
[5/30] Align món: Bún gạo xào lòng gà
[6/30] Align món: Bún gỏi dà Sóc Trăng
[7/30] Align món: Bánh khoai mì nướng bằng nồi cơm điện
[8/30] Align món: Cơm chiên bò xào
[9/30] Align món: Cách làm đậu hũ xào thịt sốt cà chua đơn giản, bắt cơm
[10/30] Align món: Gỏi cá cơm tươi
[11/30] Align món: Nấu lẩu bằng nồi cơm điện
[12/30] Align món: Canh bún bạch tuộc
[13/30] Align món: Canh ngót cá cơm
[14/30] Align món: Lẩu bò nướng hầm sả
[15/30] Align món: Gỏi bò nướng chua ngọt
[16/30] Align món: Cánh gà nướng muối ớt bằng nồi chiên không dầu
[17/30] Align món: Salad bí đỏ thịt gà nướng
[18/30] Align món: Khoai tây xào thịt bò
[19/30] Align món: Thịt lợn xào khoai tây su hào
[20/30] Align món: Nem nướng Nha Trang bằng nồi chiên không dầu
[21/30] Align món: Gỏi rau diếp cá thịt bò
[22/30] Align món: Gỏi bông súng tôm thịt

In [29]:
def compute_micro_prf(tp: int, fp: int, fn: int) -> Tuple[float, float, float]:
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1


per_dish_metrics = []

global_tp_all = global_fp_all = global_fn_all = 0
global_tp_core = global_fp_core = global_fn_core = 0

# Chuẩn bị cho quantity & category
quantity_errors = []
quantity_ape = []  # absolute percentage error

cat_stats = defaultdict(lambda: {"gt": 0, "pred": 0, "tp": 0})

for sample, align in zip(samples, semantic_alignments):
    dish_name = sample["dish_name"]
    gt_ings = sample["gt_ingredients"]
    pred_ings = sample["pred_ingredients"]

    gt_ids_all = {ing["ingredient_id"] for ing in gt_ings if ing["ingredient_id"]}
    pred_ids_all = {ing["ingredient_id"] for ing in pred_ings if ing["ingredient_id"]}

    # Map id -> ingredient object
    gt_by_id = {ing["ingredient_id"]: ing for ing in gt_ings if ing["ingredient_id"]}
    pred_by_id = {ing["ingredient_id"]: ing for ing in pred_ings if ing["ingredient_id"]}

    # Lấy matches hợp lệ (id phải tồn tại trong 2 bên)
    raw_matches = align.get("matches", [])
    valid_matches = []
    for m in raw_matches:
        gid = m.get("gt_id")
        pid = m.get("pred_id")
        if gid in gt_ids_all and pid in pred_ids_all:
            valid_matches.append((gid, pid))

    matched_gt_ids = {gid for gid, _ in valid_matches}
    matched_pred_ids = {pid for _, pid in valid_matches}

    # --- MICRO-F1 ALL ---
    num_gt = len(gt_ids_all)
    num_pred = len(pred_ids_all)

    tp_all = len(matched_gt_ids)  # mỗi GT được match xem như 1 TP
    fp_all = max(num_pred - len(matched_pred_ids), 0)
    fn_all = max(num_gt - len(matched_gt_ids), 0)

    global_tp_all += tp_all
    global_fp_all += fp_all
    global_fn_all += fn_all

    prec_all, rec_all, f1_all = compute_micro_prf(tp_all, fp_all, fn_all)

    # --- MICRO-F1 CORE ---
    core_gt_ids = {
        ing["ingredient_id"]
        for ing in gt_ings
        if ing.get("ingredient_id") and ing.get("importance", 1) >= MIN_IMPORTANCE_CORE
    }
    matched_core_gt_ids = matched_gt_ids & core_gt_ids

    tp_core = len(matched_core_gt_ids)
    # Cho bài toán "core", predicted positive vẫn là TẤT CẢ nguyên liệu dự đoán
    fp_core = max(num_pred - tp_core, 0)
    fn_core = max(len(core_gt_ids) - tp_core, 0)

    global_tp_core += tp_core
    global_fp_core += fp_core
    global_fn_core += fn_core

    prec_core, rec_core, f1_core = compute_micro_prf(tp_core, fp_core, fn_core)

    # --- Quantity (MAE & MAPE) trên các cặp đã align ---
    for gid, pid in valid_matches:
        gt_ing = gt_by_id[gid]
        pred_ing = pred_by_id[pid]

        gt_q = parse_quantity_str(gt_ing.get("quantity"))
        pred_q = parse_quantity_str(pred_ing.get("quantity"))

        if gt_q > 0:
            err = abs(gt_q - pred_q)
            quantity_errors.append(err)
            quantity_ape.append(err / gt_q * 100)

    # --- Category-level stats: dùng GT category + semantic match ---
    for ing in gt_ings:
        cat = ing.get("category") or "unknown"
        cat_stats[cat]["gt"] += 1

    for ing in pred_ings:
        cat = ing.get("category") or "unknown"
        cat_stats[cat]["pred"] += 1

    for gid, _ in valid_matches:
        cat = gt_by_id[gid].get("category") or "unknown"
        cat_stats[cat]["tp"] += 1

    # Lưu per-dish
    per_dish_metrics.append(
        {
            "dish_name": dish_name,
            "num_gt": num_gt,
            "num_pred": num_pred,
            "tp_all": tp_all,
            "fp_all": fp_all,
            "fn_all": fn_all,
            "precision_all": prec_all,
            "recall_all": rec_all,
            "f1_all": f1_all,
            "core_gt_count": len(core_gt_ids),
            "tp_core": tp_core,
            "fp_core": fp_core,
            "fn_core": fn_core,
            "precision_core": prec_core,
            "recall_core": rec_core,
            "f1_core": f1_core,
        }
    )

# --- Global Micro-F1 ---
micro_precision_all, micro_recall_all, micro_f1_all = compute_micro_prf(
    global_tp_all, global_fp_all, global_fn_all
)
micro_precision_core, micro_recall_core, micro_f1_core = compute_micro_prf(
    global_tp_core, global_fp_core, global_fn_core
)

print("=" * 80)
print("INGREDIENT-LEVEL (SEMANTIC) — MICRO-F1 ALL INGREDIENTS")
print("=" * 80)
print(f"Precision_all: {micro_precision_all:.4f} ({micro_precision_all*100:.2f}%)")
print(f"Recall_all:    {micro_recall_all:.4f} ({micro_recall_all*100:.2f}%)")
print(f"F1_all:        {micro_f1_all:.4f} ({micro_f1_all*100:.2f}%)")

print("\n" + "=" * 80)
print(f"INGREDIENT-LEVEL (SEMANTIC) — CORE INGREDIENTS (importance ≥ {MIN_IMPORTANCE_CORE})")
print("=" * 80)
print(f"Precision_core: {micro_precision_core:.4f} ({micro_precision_core*100:.2f}%)")
print(f"Recall_core:    {micro_recall_core:.4f} ({micro_recall_core*100:.2f}%)")
print(f"F1_core:        {micro_f1_core:.4f} ({micro_f1_core*100:.2f}%)")


INGREDIENT-LEVEL (SEMANTIC) — MICRO-F1 ALL INGREDIENTS
Precision_all: 0.9119 (91.19%)
Recall_all:    0.8523 (85.23%)
F1_all:        0.8811 (88.11%)

INGREDIENT-LEVEL (SEMANTIC) — CORE INGREDIENTS (importance ≥ 2)
Precision_core: 0.6192 (61.92%)
Recall_core:    0.8628 (86.28%)
F1_core:        0.7210 (72.10%)


In [30]:
if quantity_errors:
    mae = float(np.mean(quantity_errors))
    mape = float(np.mean(quantity_ape))
else:
    mae = 0.0
    mape = 0.0

print("=" * 80)
print("SLOT-LEVEL — QUANTITY (dựa trên SEMANTIC matches)")
print("=" * 80)
print(f"MAE:  {mae:.4f}")
print(f"MAPE: {mape:.2f}%")
print(f"Số cặp quantity được evaluate: {len(quantity_errors)}")


SLOT-LEVEL — QUANTITY (dựa trên SEMANTIC matches)
MAE:  6.7955
MAPE: 197.92%
Số cặp quantity được evaluate: 352


In [31]:
category_metrics = {}

for cat, st in cat_stats.items():
    tp = st["tp"]
    gt = st["gt"]
    pred = st["pred"]
    fp = max(pred - tp, 0)
    fn = max(gt - tp, 0)

    p, r, f = compute_micro_prf(tp, fp, fn)
    category_metrics[cat] = {
        "precision": p,
        "recall": r,
        "f1": f,
        "total_gt": gt,
        "total_pred": pred,
    }

print("=" * 80)
print("CATEGORY-LEVEL (SEMANTIC) F1")
print("=" * 80)
print(f"{'Category':<30} {'Precision':>10} {'Recall':>10} {'F1':>10} {'GT':>6} {'Pred':>6}")
print("-" * 80)

for cat in sorted(category_metrics.keys(), key=lambda c: category_metrics[c]["f1"], reverse=True):
    m = category_metrics[cat]
    print(
        f"{cat:<30} "
        f"{m['precision']:>10.4f} "
        f"{m['recall']:>10.4f} "
        f"{m['f1']:>10.4f} "
        f"{m['total_gt']:>6} "
        f"{m['total_pred']:>6}"
    )

print("=" * 80)


CATEGORY-LEVEL (SEMANTIC) F1
Category                        Precision     Recall         F1     GT   Pred
--------------------------------------------------------------------------------
milk                               1.0000     1.0000     1.0000      2      0
cold_cuts:_sausages_&_ham          1.0000     1.0000     1.0000      4      3
dried_fruits                       1.0000     1.0000     1.0000      1      1
snacks                             1.0000     0.9231     0.9600     13     12
seafood_&_fish_balls               1.0000     0.8750     0.9333     16     14
grains_staples                     1.0000     0.8571     0.9231     21     15
seasonings                         1.0000     0.8293     0.9067    164    129
vegetables                         0.9118     0.8942     0.9029    104    102
fresh_meat                         0.9286     0.8667     0.8966     30     28
others                             0.7609     0.8333     0.7955     42     46
alcoholic_beverages             

In [32]:
per_dish_df = pd.DataFrame(per_dish_metrics)
per_dish_df.to_csv("semantic_f1_per_dish.csv", index=False, encoding="utf-8-sig")

print("💾 Đã lưu 'semantic_f1_per_dish.csv'")

total_fp_all = int(per_dish_df["fp_all"].sum())
total_fn_all = int(per_dish_df["fn_all"].sum())
perfect_matches = int((per_dish_df["f1_all"] == 1.0).sum())

summary_report = {
    "timestamp": pd.Timestamp.now().isoformat(),
    "validation": {
        "original_sample_size": int(original_sample_size),
        "valid_predictions": int(len(samples)),
        "excluded_dishes_count": int(len(excluded_dishes)),
        "excluded_dishes": excluded_dishes,
        "failed_predictions": int(len(errors)),
        "retention_rate": float(len(samples) / original_sample_size) if original_sample_size else 1.0,
    },
    "test_dataset_size": int(len(samples)),
    "ingredient_level": {
        # Lưu ý: các metric này BÂY GIỜ là SEMANTIC (LLM-based)
        "micro_f1_all": float(micro_f1_all),
        "precision_all": float(micro_precision_all),
        "recall_all": float(micro_recall_all),
        "micro_f1_core": float(micro_f1_core),
        "precision_core": float(micro_precision_core),
        "recall_core": float(micro_recall_core),
    },
    "slot_level": {
        "quantity_mae": float(mae),
        "quantity_mape": float(mape),
    },
    "category_level": {
        cat: {
            "precision": float(m["precision"]),
            "recall": float(m["recall"]),
            "f1": float(m["f1"]),
            "total_gt": int(m["total_gt"]),
            "total_pred": int(m["total_pred"]),
        }
        for cat, m in category_metrics.items()
    },
    "error_statistics": {
        "total_fp_all": total_fp_all,
        "total_fn_all": total_fn_all,
        "perfect_matches": perfect_matches,
    },
}

with open("semantic_f1_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_report, f, indent=2, ensure_ascii=False)

print("💾 Đã lưu 'semantic_f1_summary.json'")


💾 Đã lưu 'semantic_f1_per_dish.csv'
💾 Đã lưu 'semantic_f1_summary.json'
